In [2]:
!pip install youtube-transcript-api langchain-text-splitters langchain-google-genai langchain-community faiss-cpu
# !pip install -q langchain-openai
!pip install -q langchain-google-genai
!pip install -q langchain-huggingface sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.1/591.1 kB 17.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 81.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 67.6 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.20
    Uninstalling langchain-protocol-0.0.20:
      Successfully uninstalled langchain-protocol-0.0.20
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-c

In [1]:
#youtube chatbot
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("enter api key ")

enter api key ··········


In [2]:
from youtube_transcript_api import( YouTubeTranscriptApi, TranscriptsDisabled)

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)

from langchain_community.vectorstores import FAISS

from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_core.prompts import PromptTemplate
print("All libraries imported successfully!")


All libraries imported successfully!


/tmp/ipykernel_21198/3900562805.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
#step 1 indexing
video_id="LPZh9BOjkQs"

transcript_list= [{'text': 'Imagine you happen across a short movie script that',
  'start': 1.14,
  'duration': 2.836},
 {'text': 'describes a scene between a person and their AI assistant.',
  'start': 3.976,
  'duration': 3.164},
 {'text': "The script has what the person asks the AI, but the AI's response has been torn off.",
  'start': 7.48,
  'duration': 5.58},
 {'text': 'Suppose you also have this powerful magical machine that can take',
  'start': 13.06,
  'duration': 3.92},
 {'text': 'any text and provide a sensible prediction of what word comes next.',
  'start': 16.98,
  'duration': 3.98},
 {'text': 'You could then finish the script by feeding in what you have to the machine,',
  'start': 21.5,
  'duration': 4.006},
 {'text': "seeing what it would predict to start the AI's answer,",
  'start': 25.506,
  'duration': 2.862},
 {'text': 'and then repeating this over and over with a growing script completing the dialogue.',
  'start': 28.368,
  'duration': 4.372},
 {'text': "When you interact with a chatbot, this is exactly what's happening.",
  'start': 33.38,
  'duration': 3.1},
 {'text': 'A large language model is a sophisticated mathematical function',
  'start': 37.02,
  'duration': 3.681},
 {'text': 'that predicts what word comes next for any piece of text.',
  'start': 40.701,
  'duration': 3.279},
 {'text': 'Instead of predicting one word with certainty, though,',
  'start': 44.38,
  'duration': 3.022},
 {'text': 'what it does is assign a probability to all possible next words.',
  'start': 47.402,
  'duration': 3.518},
 {'text': 'To build a chatbot, you lay out some text that describes an interaction between a user',
  'start': 51.62,
  'duration': 5.18},
 {'text': 'and a hypothetical AI assistant, add on whatever the user types in as the first part of',
  'start': 56.8,
  'duration': 5.24},
 {'text': 'the interaction, and then have the model repeatedly predict the next word that such a',
  'start': 62.04,
  'duration': 5.12},
 {'text': "hypothetical AI assistant would say in response, and that's what's presented to the user.",
  'start': 67.16,
  'duration': 5.3},
 {'text': 'In doing this, the output tends to look a lot more natural if',
  'start': 73.08,
  'duration': 3.134},
 {'text': 'you allow it to select less likely words along the way at random.',
  'start': 76.214,
  'duration': 3.286},
 {'text': 'So what this means is even though the model itself is deterministic,',
  'start': 80.14,
  'duration': 3.48},
 {'text': "a given prompt typically gives a different answer each time it's run.",
  'start': 83.62,
  'duration': 3.48},
 {'text': 'Models learn how to make these predictions by processing an enormous amount of text,',
  'start': 88.04,
  'duration': 4.292},
 {'text': 'typically pulled from the internet.',
  'start': 92.332,
  'duration': 1.768},
 {'text': 'For a standard human to read the amount of text that was used to train GPT-3,',
  'start': 94.1,
  'duration': 5.371},
 {'text': 'for example, if they read non-stop 24-7, it would take over 2600 years.',
  'start': 99.471,
  'duration': 4.889},
 {'text': 'Larger models since then train on much, much more.',
  'start': 104.72,
  'duration': 2.62},
 {'text': 'You can think of training a little bit like tuning the dials on a big machine.',
  'start': 108.2,
  'duration': 3.58},
 {'text': 'The way that a language model behaves is entirely determined by these',
  'start': 112.28,
  'duration': 4.021},
 {'text': 'many different continuous values, usually called parameters or weights.',
  'start': 116.301,
  'duration': 4.079},
 {'text': 'Changing those parameters will change the probabilities',
  'start': 121.02,
  'duration': 3.079},
 {'text': 'that the model gives for the next word on a given input.',
  'start': 124.099,
  'duration': 3.081},
 {'text': 'What puts the large in large language model is how',
  'start': 127.86,
  'duration': 2.867},
 {'text': 'they can have hundreds of billions of these parameters.',
  'start': 130.727,
  'duration': 3.093},
 {'text': 'No human ever deliberately sets those parameters.',
  'start': 135.2,
  'duration': 2.84},
 {'text': 'Instead, they begin at random, meaning the model just outputs gibberish,',
  'start': 138.44,
  'duration': 4.203},
 {'text': "but they're repeatedly refined based on many example pieces of text.",
  'start': 142.643,
  'duration': 3.917},
 {'text': 'One of these training examples could be just a handful of words,',
  'start': 147.14,
  'duration': 3.516},
 {'text': 'or it could be thousands, but in either case, the way this works is to',
  'start': 150.656,
  'duration': 3.84},
 {'text': 'pass in all but the last word from that example into the model and',
  'start': 154.496,
  'duration': 3.624},
 {'text': 'compare the prediction that it makes with the true last word from the example.',
  'start': 158.12,
  'duration': 4.22},
 {'text': 'An algorithm called backpropagation is used to tweak all of the parameters',
  'start': 163.26,
  'duration': 4.133},
 {'text': 'in such a way that it makes the model a little more likely to choose',
  'start': 167.393,
  'duration': 3.803},
 {'text': 'the true last word and a little less likely to choose all the others.',
  'start': 171.196,
  'duration': 3.804},
 {'text': 'When you do this for many, many trillions of examples,',
  'start': 175.74,
  'duration': 3.01},
 {'text': 'not only does the model start to give more accurate predictions on the training data,',
  'start': 178.75,
  'duration': 4.708},
 {'text': "but it also starts to make more reasonable predictions on text that it's never",
  'start': 183.458,
  'duration': 4.325},
 {'text': 'seen before.', 'start': 187.783, 'duration': 0.657},
 {'text': 'Given the huge number of parameters and the enormous amount of training data,',
  'start': 189.42,
  'duration': 4.499},
 {'text': 'the scale of computation involved in training a large language model is mind-boggling.',
  'start': 193.919,
  'duration': 4.961},
 {'text': 'To illustrate, imagine that you could perform one',
  'start': 199.6,
  'duration': 2.685},
 {'text': 'billion additions and multiplications every single second.',
  'start': 202.285,
  'duration': 3.115},
 {'text': 'How long do you think it would take for you to do all of the',
  'start': 206.06,
  'duration': 3.266},
 {'text': 'operations involved in training the largest language models?',
  'start': 209.326,
  'duration': 3.214},
 {'text': 'Do you think it would take a year?',
  'start': 213.46,
  'duration': 1.579},
 {'text': 'Maybe something like 10,000 years?',
  'start': 216.039,
  'duration': 1.921},
 {'text': 'The answer is actually much more than that.',
  'start': 219.02,
  'duration': 1.78},
 {'text': "It's well over 100 million years.",
  'start': 221.12,
  'duration': 2.78},
 {'text': 'This is only part of the story, though.',
  'start': 225.52,
  'duration': 1.84},
 {'text': 'This whole process is called pre-training.',
  'start': 227.54,
  'duration': 1.68},
 {'text': 'The goal of auto-completing a random passage of text from the',
  'start': 229.5,
  'duration': 3.146},
 {'text': 'internet is very different from the goal of being a good AI assistant.',
  'start': 232.646,
  'duration': 3.554},
 {'text': 'To address this, chatbots undergo another type of training,',
  'start': 236.88,
  'duration': 3.2},
 {'text': 'just as important, called reinforcement learning with human feedback.',
  'start': 240.08,
  'duration': 3.68},
 {'text': 'Workers flag unhelpful or problematic predictions,',
  'start': 244.48,
  'duration': 3.018},
 {'text': "and their corrections further change the model's parameters,",
  'start': 247.498,
  'duration': 3.611},
 {'text': 'making them more likely to give predictions that users prefer.',
  'start': 251.109,
  'duration': 3.671},
 {'text': 'Looking back at the pre-training, though, this staggering amount of',
  'start': 254.78,
  'duration': 4.08},
 {'text': 'computation is only made possible by using special computer chips that',
  'start': 258.86,
  'duration': 4.26},
 {'text': 'are optimized for running many operations in parallel, known as GPUs.',
  'start': 263.12,
  'duration': 4.14},
 {'text': 'However, not all language models can be easily parallelized.',
  'start': 268.12,
  'duration': 3.5},
 {'text': 'Prior to 2017, most language models would process text one word at a time,',
  'start': 272.08,
  'duration': 4.737},
 {'text': 'but then a team of researchers at Google introduced a new model known as the transformer.',
  'start': 276.817,
  'duration': 5.623},
 {'text': "Transformers don't read text from the start to the finish,",
  'start': 283.3,
  'duration': 3.445},
 {'text': 'they soak it all in at once, in parallel.',
  'start': 286.745,
  'duration': 2.395},
 {'text': 'The very first step inside a transformer, and most other language models for that matter,',
  'start': 289.9,
  'duration': 4.7},
 {'text': 'is to associate each word with a long list of numbers.',
  'start': 294.6,
  'duration': 2.82},
 {'text': 'The reason for this is that the training process only works with continuous values,',
  'start': 297.86,
  'duration': 4.536},
 {'text': 'so you have to somehow encode language using numbers,',
  'start': 302.396,
  'duration': 2.916},
 {'text': 'and each of these lists of numbers may somehow encode the meaning of the',
  'start': 305.312,
  'duration': 3.942},
 {'text': 'corresponding word.', 'start': 309.254, 'duration': 1.026},
 {'text': 'What makes transformers unique is their reliance',
  'start': 310.28,
  'duration': 3.08},
 {'text': 'on a special operation known as attention.',
  'start': 313.36,
  'duration': 2.64},
 {'text': 'This operation gives all of these lists of numbers a chance to talk to one another',
  'start': 316.98,
  'duration': 4.704},
 {'text': 'and refine the meanings they encode based on the context around, all done in parallel.',
  'start': 321.684,
  'duration': 4.876},
 {'text': 'For example, the numbers encoding the word bank might be changed based on the',
  'start': 327.4,
  'duration': 4.307},
 {'text': 'context surrounding it to somehow encode the more specific notion of a riverbank.',
  'start': 331.707,
  'duration': 4.473},
 {'text': 'Transformers typically also include a second type of operation known',
  'start': 337.28,
  'duration': 3.749},
 {'text': 'as a feed-forward neural network, and this gives the model extra',
  'start': 341.029,
  'duration': 3.532},
 {'text': 'capacity to store more patterns about language learned during training.',
  'start': 344.561,
  'duration': 3.859},
 {'text': 'All of this data repeatedly flows through many different iterations of',
  'start': 349.28,
  'duration': 4.121},
 {'text': 'these two fundamental operations, and as it does so,',
  'start': 353.401,
  'duration': 3.077},
 {'text': 'the hope is that each list of numbers is enriched to encode whatever',
  'start': 356.478,
  'duration': 4.006},
 {'text': 'information might be needed to make an accurate prediction of what word',
  'start': 360.484,
  'duration': 4.18},
 {'text': 'follows in the passage.', 'start': 364.664, 'duration': 1.336},
 {'text': 'At the end, one final function is performed on the last vector in this sequence,',
  'start': 367.0,
  'duration': 4.534},
 {'text': 'which now has had a chance to be influenced by all the other context from the input text,',
  'start': 371.534,
  'duration': 5.039},
 {'text': 'as well as everything the model learned during training,',
  'start': 376.573,
  'duration': 3.191},
 {'text': 'to produce a prediction of the next word.',
  'start': 379.764,
  'duration': 2.296},
 {'text': "Again, the model's prediction looks like a probability for every possible next word.",
  'start': 382.48,
  'duration': 4.88},
 {'text': 'Although researchers design the framework for how each of these steps work,',
  'start': 388.56,
  'duration': 4.234},
 {'text': "it's important to understand that the specific behavior is an emergent phenomenon",
  'start': 392.794,
  'duration': 4.568},
 {'text': 'based on how those hundreds of billions of parameters are tuned during training.',
  'start': 397.362,
  'duration': 4.458},
 {'text': 'This makes it incredibly challenging to determine',
  'start': 402.48,
  'duration': 2.59},
 {'text': 'why the model makes the exact predictions that it does.',
  'start': 405.07,
  'duration': 2.85},
 {'text': 'What you can see is that when you use large language model predictions to autocomplete',
  'start': 408.44,
  'duration': 5.338},
 {'text': 'a prompt, the words that it generates are uncannily fluent, fascinating, and even useful.',
  'start': 413.778,
  'duration': 5.462},
 {'text': "If you're a new viewer and you're curious about more details on how",
  'start': 425.719,
  'duration': 3.107},
 {'text': 'transformers and attention work, boy do I have some material for you.',
  'start': 428.826,
  'duration': 3.153},
 {'text': 'One option is to jump into a series I made about deep learning,',
  'start': 432.399,
  'duration': 3.681},
 {'text': 'where we visualize and motivate the details of attention and all the other steps',
  'start': 436.08,
  'duration': 4.66},
 {'text': 'in a transformer.', 'start': 440.74, 'duration': 0.979},
 {'text': 'Also, on my second channel I just posted a talk I gave a couple',
  'start': 442.099,
  'duration': 3.43},
 {'text': 'months ago about this topic for the company TNG in Munich.',
  'start': 445.529,
  'duration': 3.11},
 {'text': 'Sometimes I actually prefer the content I make as a casual talk rather than a produced',
  'start': 449.079,
  'duration': 4.022},
 {'text': 'video, but I leave it up to you which one of these feels like the better follow-on.',
  'start': 453.101,
  'duration': 3.838}]

  # transcript=" ".join(chunk.text for chunk in transcript_list.snippets)
transcript =" ".join( chunk["text"] if isinstance(chunk,dict) else chunk.text for chunk in transcript_list)


print(transcript)
print("\nTotal characters:",len(transcript))




Imagine you happen across a short movie script that describes a scene between a person and their AI assistant. The script has what the person asks the AI, but the AI's response has been torn off. Suppose you also have this powerful magical machine that can take any text and provide a sensible prediction of what word comes next. You could then finish the script by feeding in what you have to the machine, seeing what it would predict to start the AI's answer, and then repeating this over and over with a growing script completing the dialogue. When you interact with a chatbot, this is exactly what's happening. A large language model is a sophisticated mathematical function that predicts what word comes next for any piece of text. Instead of predicting one word with certainty, though, what it does is assign a probability to all possible next words. To build a chatbot, you lay out some text that describes an interaction between a user and a hypothetical AI assistant, add on whatever the use

In [4]:
#step 1b- Indexing(text splitting)
splitter= RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
chunks = splitter.create_documents([transcript])
print("Total chunks:", len(chunks))

Total chunks: 19


In [ ]:
#step 1c,1d embedding generation and storing in vector store

In [7]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vector_store = FAISS.from_documents(
    chunks,
    embeddings
)

print("Executed successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Executed successfully!


In [8]:
# vector_store.get_by_ids([])#id daal from above cell ouput

In [9]:
from re import search
#step 2 Retrieval
retriever = vector_store.as_retriever(search_type="similarity",search_kwargs={"k":4})



In [10]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x792caa6f12b0>, search_kwargs={'k': 4})

In [11]:
retriever.invoke("what is deepmind")#retriever ko input m query aur output m list of files

[Document(id='2f696ba1-bacd-4943-bbbd-bf0f586ca8e6', metadata={}, page_content='known as attention. This operation gives all of these lists of numbers a chance to talk to one another and refine the meanings they encode based on the context around, all done in parallel. For example, the numbers encoding the word bank might be changed based on the context surrounding it to somehow encode the more specific notion of a riverbank. Transformers typically also include a second type of operation known as a feed-forward neural network, and this gives the model extra capacity to'),
 Document(id='d6da8781-5533-48a7-92d5-7b40d0611173', metadata={}, page_content="reinforcement learning with human feedback. Workers flag unhelpful or problematic predictions, and their corrections further change the model's parameters, making them more likely to give predictions that users prefer. Looking back at the pre-training, though, this staggering amount of computation is only made possible by using special com

In [12]:
query = "How does an AI predict the next word?"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):
    print(f"\nChunk {i+1}:")
    print(doc.page_content)


Chunk 1:
prediction of what word follows in the passage. At the end, one final function is performed on the last vector in this sequence, which now has had a chance to be influenced by all the other context from the input text, as well as everything the model learned during training, to produce a prediction of the next word. Again, the model's prediction looks like a probability for every possible next word. Although researchers design the framework for how each of these steps work, it's important to

Chunk 2:
Imagine you happen across a short movie script that describes a scene between a person and their AI assistant. The script has what the person asks the AI, but the AI's response has been torn off. Suppose you also have this powerful magical machine that can take any text and provide a sensible prediction of what word comes next. You could then finish the script by feeding in what you have to the machine, seeing what it would predict to start the AI's answer, and then repeating th

In [13]:
#step 3 Augmentation
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.2
)

In [14]:
prompt= PromptTemplate(
    template="""
            You are a helpful assistant.
            Answer ONLY from the  provided transcript context.
            If the context is insufficient,just say you don't know.
            {context}
            Question:{question}
            """,
    input_variables =['context','question']
)

In [15]:
question ="How does an AI predict the next word?"
retrieved_docs = retriever.invoke(question)

In [34]:
retrieved_docs

[Document(id='5f223cbc-cf13-4507-972e-6d83f833cb08', metadata={}, page_content='steps in a transformer. Also, on my second channel I just posted a talk I gave a couple months ago about this topic for the company TNG in Munich. Sometimes I actually prefer the content I make as a casual talk rather than a produced video, but I leave it up to you which one of these feels like the better follow-on.'),
 Document(id='df760c7b-fc69-4b25-b18d-496ccd58331e', metadata={}, page_content="model predictions to autocomplete a prompt, the words that it generates are uncannily fluent, fascinating, and even useful. If you're a new viewer and you're curious about more details on how transformers and attention work, boy do I have some material for you. One option is to jump into a series I made about deep learning, where we visualize and motivate the details of attention and all the other steps in a transformer. Also, on my second channel I just posted a talk I gave a couple months ago"),
 Document(id='e7

In [19]:
context_text="\n\n".join(doc.page_content for doc in retrieved_docs)
context_text
#har documentske pass jaa rahe uska page content utha rahe h  aur usko join kr rahe h

"prediction of what word follows in the passage. At the end, one final function is performed on the last vector in this sequence, which now has had a chance to be influenced by all the other context from the input text, as well as everything the model learned during training, to produce a prediction of the next word. Again, the model's prediction looks like a probability for every possible next word. Although researchers design the framework for how each of these steps work, it's important to\n\nImagine you happen across a short movie script that describes a scene between a person and their AI assistant. The script has what the person asks the AI, but the AI's response has been torn off. Suppose you also have this powerful magical machine that can take any text and provide a sensible prediction of what word comes next. You could then finish the script by feeding in what you have to the machine, seeing what it would predict to start the AI's answer, and then repeating this over and over

In [17]:
final_prompt = prompt.invoke({"context": context_text,"question":question})

In [37]:
final_prompt

StringPromptValue(text="\n            You are a helpful assistant.\n            Answer ONLY from the  provided transcript context.\n            If the context is insufficient,just say you don't know.\n            steps in a transformer. Also, on my second channel I just posted a talk I gave a couple months ago about this topic for the company TNG in Munich. Sometimes I actually prefer the content I make as a casual talk rather than a produced video, but I leave it up to you which one of these feels like the better follow-on.\n\nmodel predictions to autocomplete a prompt, the words that it generates are uncannily fluent, fascinating, and even useful. If you're a new viewer and you're curious about more details on how transformers and attention work, boy do I have some material for you. One option is to jump into a series I made about deep learning, where we visualize and motivate the details of attention and all the other steps in a transformer. Also, on my second channel I just posted 

In [18]:
# step 4 :Generation
# from google import genai

# client = genai.Client()

# for model in client.models.list():
#     print(model.name)
try:
    answer = llm.invoke(final_prompt)
    print(answer.content)

except Exception as e:
    print("Generation failed:", e)


# import time
# for attempt in range(3):
#   try:
#     answer=llm.invoke(final_prompt)
#     print(answer.content)
#     break
#   except Exception as e:
#     print("Error:",e)
#     print("Retrying in 30 seconds...")
#     time.sleep(30)
#You can use this to retry temporary server errors:

[{'type': 'text', 'text': 'The AI predicts the next word by acting as a sophisticated mathematical function that assigns a probability to all possible next words rather than predicting one word with certainty. It processes the input text, allowing the final vector to be influenced by the context and everything the model learned during training, to produce a prediction of the next word.', 'extras': {'signature': 'EqEBCp4BAWkUfROP+F1EzAO2zgJaYdXxp/37ctcACoUqULoHvVkNnSSNriXmUNIp736Cy74ckOse/QHpRegf3u8QHZI9QwNUZMCs08/qzAIODDtF/o9JWlA8QaCfCtjzGaeCJTE0kNMz9d89XDpnl7VXgKSOnz1uHLTxs8jQOTa2HCtfDnWNS7tg1F2sgYTi5Wz4wv080WuVQEpKllWlv3BwPpU='}}]


In [39]:
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Document {i+1} ---")
    print(doc.page_content)


--- Document 1 ---
steps in a transformer. Also, on my second channel I just posted a talk I gave a couple months ago about this topic for the company TNG in Munich. Sometimes I actually prefer the content I make as a casual talk rather than a produced video, but I leave it up to you which one of these feels like the better follow-on.

--- Document 2 ---
model predictions to autocomplete a prompt, the words that it generates are uncannily fluent, fascinating, and even useful. If you're a new viewer and you're curious about more details on how transformers and attention work, boy do I have some material for you. One option is to jump into a series I made about deep learning, where we visualize and motivate the details of attention and all the other steps in a transformer. Also, on my second channel I just posted a talk I gave a couple months ago

--- Document 3 ---
prediction of what word follows in the passage. At the end, one final function is performed on the last vector in this seq

In [20]:
#building a chain
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

def format_docs(retrieved_docs):#function banaya jisko hum retrieved docs denge
  context_text="\n\n".join(doc.page_content for doc in retrieved_docs)
  #aur ye function  un sbh document  ki string ko nikalega aur unko concatinate krke retuen krdega
  return context_text

In [21]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),#retriever ko q mila or ye palt ke list of documents dera pr unko context m nhi de sakte toh unko thora process kr rahe
    'question': RunnablePassthrough()
})

In [22]:
parallel_chain.invoke('who is demis')

{'context': "in parallel, known as GPUs. However, not all language models can be easily parallelized. Prior to 2017, most language models would process text one word at a time, but then a team of researchers at Google introduced a new model known as the transformer. Transformers don't read text from the start to the finish, they soak it all in at once, in parallel. The very first step inside a transformer, and most other language models for that matter, is to associate each word with a long list of numbers.\n\nsteps in a transformer. Also, on my second channel I just posted a talk I gave a couple months ago about this topic for the company TNG in Munich. Sometimes I actually prefer the content I make as a casual talk rather than a produced video, but I leave it up to you which one of these feels like the better follow-on.\n\nlike 10,000 years? The answer is actually much more than that. It's well over 100 million years. This is only part of the story, though. This whole process is call

In [24]:
parser =StrOutputParser()#q ki output string format m dekhna h

In [25]:
main_chain = parallel_chain|prompt|llm|parser#paralle chain se 2 input niklega vo prompt ko milega phir  1ouput vo llm ko milega aur llm se ohir parser ko milega

In [26]:
main_chain.invoke("can you summarize the video")

'Based on the provided transcript, the video discusses how transformer models work, specifically focusing on the process of predicting the next word in a passage. Key points include:\n\n*   **Core Function:** The model uses a series of steps to process input text and generate a probability distribution for every possible next word.\n*   **Fundamental Operations:** Data flows through iterations of two main operations: attention mechanisms and feed-forward neural networks. The latter provides extra capacity to store language patterns learned during training.\n*   **Goal:** Through these repeated operations, lists of numbers are enriched to encode information necessary for making accurate predictions.\n*   **Final Step:** A final function is performed on the last vector in the sequence—which has been influenced by the input context and training data—to produce the final word prediction.'